# Strace log → per-file-path syscall table

Reads a strace-style log (e.g. `out.log`). Each **row** is one filesystem path that appears in the trace. **Columns** are:

- One count column per **syscall** (`openat`, `newfstatat`, …).
- **`total_syscall`**: how many syscalls referenced that path.
- **`errno`** / **`errormessage`**: distinct errors seen for that path (when the kernel returned an errno).
- **`sample_line`**: first log line number for that path (for cross-checking `out.log`).

Only syscalls whose **primary quoted path** starts with `PATH_PREFIX` (default `/home/anurag/opensource/TEST`) are counted.


In [1]:
import re
from pathlib import Path

import pandas as pd

_ERRNO_RE = re.compile(r"^(-?\d+)\s+([A-Z0-9_]+)\s+\((.*)\)\s*$")
_INT_RE = re.compile(r"^-?\d+$")
_QUOTED_ABS = re.compile(r'"(/[^"]*)"')


def _split_syscall_and_args(left: str):
    try:
        i = left.index("(")
    except ValueError:
        return None, None
    name = left[:i].strip()
    depth = 0
    for j in range(i, len(left)):
        c = left[j]
        if c == "(":
            depth += 1
        elif c == ")":
            depth -= 1
            if depth == 0:
                return name, left[i + 1 : j]
    return name, left[i + 1 :]


def parse_strace_line(line: str) -> dict:
    line = line.rstrip("\n\r")
    row = {"raw": line}
    if " = " not in line:
        row["kind"] = "message" if line.strip() else "blank"
        row["syscall"] = None
        row["arguments"] = None
        row["result_raw"] = None
        row["errno"] = None
        row["errno_message"] = None
        return row
    left, right = line.rsplit(" = ", 1)
    right = right.strip()
    syscall, args = _split_syscall_and_args(left)
    row["kind"] = "syscall"
    row["syscall"] = syscall
    row["arguments"] = args
    m = _ERRNO_RE.match(right)
    if m:
        row["result_raw"] = right
        row["errno"] = m.group(2)
        row["errno_message"] = m.group(3)
        return row
    if _INT_RE.match(right):
        row["result_raw"] = right
        row["errno"] = None
        row["errno_message"] = None
        return row
    row["result_raw"] = right
    row["errno"] = None
    row["errno_message"] = None
    return row


def extract_file_path(left_part: str) -> str | None:
    """First quoted absolute path in the syscall text (before ' = ')."""
    m = _QUOTED_ABS.search(left_part)
    return m.group(1) if m else None


def log_to_syscall_rows(path: Path) -> pd.DataFrame:
    text = path.read_text(encoding="utf-8", errors="replace")
    parsed = [parse_strace_line(ln) for ln in text.splitlines()]
    df = pd.DataFrame(parsed)
    df.insert(0, "line", range(1, len(df) + 1))
    return df


def filepath_syscall_summary(
    df: pd.DataFrame, *, path_prefix: str | None = None
) -> pd.DataFrame:
    """Wide table: file_path, <syscall counts>, total_syscall, errno, errormessage, sample_line."""
    d = df.loc[df["kind"] == "syscall"].copy()
    d = d[d["syscall"].notna()]
    d["file_path"] = d["raw"].astype(str).map(
        lambda s: extract_file_path(s.rsplit(" = ", 1)[0]) if " = " in s else None
    )
    d = d.loc[d["file_path"].notna()]
    if path_prefix:
        d = d.loc[d["file_path"].str.startswith(path_prefix, na=False)]
    if d.empty:
        return pd.DataFrame()

    ctab = pd.crosstab(d["file_path"], d["syscall"])
    syscall_cols = sorted(ctab.columns, key=str)
    ctab = ctab.reindex(columns=syscall_cols, fill_value=0)
    ctab.index.name = "file_path"
    totals = d.groupby("file_path", sort=False).size().rename("total_syscall")
    first_line = d.groupby("file_path", sort=False)["line"].min().rename("sample_line")

    err_subset = d.loc[
        d["errno"].notna(), ["file_path", "errno", "errno_message"]
    ].drop_duplicates()
    if err_subset.empty:
        errs = pd.DataFrame(index=ctab.index, columns=["errno", "errormessage"])
        errs["errno"] = ""
        errs["errormessage"] = ""
    else:
        errs = err_subset.groupby("file_path", sort=False).agg(
            errno=(
                "errno",
                lambda s: "; ".join(dict.fromkeys(s.astype(str))),
            ),
            errormessage=(
                "errno_message",
                lambda s: "; ".join(
                    dict.fromkeys(x for x in s.dropna().astype(str) if x)
                ),
            ),
        )

    out = (
        ctab.join(totals)
        .join(first_line)
        .join(errs, how="left")
        .reset_index()
    )
    out["errno"] = out["errno"].fillna("")
    out["errormessage"] = out["errormessage"].fillna("")
    meta = ["file_path"] + syscall_cols + ["total_syscall", "errno", "errormessage", "sample_line"]
    meta = [c for c in meta if c in out.columns]
    return out[meta]

In [2]:
HERE = Path.cwd()
LOG_PATH = HERE / "out.log"
PATH_PREFIX = "/home/anurag/opensource/TEST"

if not LOG_PATH.is_file():
    raise FileNotFoundError(
        f"out.log not found at {LOG_PATH.resolve()}. "
        "cd into the folder that contains it or set LOG_PATH."
    )

raw_df = log_to_syscall_rows(LOG_PATH)
summary = filepath_syscall_summary(raw_df, path_prefix=PATH_PREFIX)
print(f"Log lines: {len(raw_df)} | Paths under {PATH_PREFIX!r}: {len(summary)}")
sc = [c for c in summary.columns if c not in ("file_path", "total_syscall", "errno", "errormessage", "sample_line")]
print(f"Syscall columns ({len(sc)}): {', '.join(sc)}")

Log lines: 2862 | Paths under '/home/anurag/opensource/TEST': 897
Syscall columns (7): access, getcwd, mkdir, newfstatat, openat, rename, unlink


In [3]:
from IPython.display import display

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

display(summary.sort_values("total_syscall", ascending=False).head(40))

,file_path,access,getcwd,mkdir,newfstatat,openat,rename,unlink,total_syscall,errno,errormessage,sample_line
237,/home/anurag/opensource/TEST/dulwich-1.1.0/.git/refs/heads/master,0,0,0,0,5,0,0,5,,,1228
29,/home/anurag/opensource/TEST/dulwich-1.1.0/.git/objects/0f,0,0,5,0,0,0,0,5,EEXIST,File exists,2508
12,/home/anurag/opensource/TEST/dulwich-1.1.0/.git/index,0,0,0,2,2,0,0,4,,,1240
10,/home/anurag/opensource/TEST/dulwich-1.1.0/.git/commondir,0,0,0,0,4,0,0,4,ENOENT,No such file or directory,1207
88,/home/anurag/opensource/TEST/dulwich-1.1.0/.git/objects/51,0,0,4,0,0,0,0,4,EEXIST,File exists,2490
508,/home/anurag/opensource/TEST/dulwich-1.1.0/out.log,0,0,0,3,1,0,0,4,,,2103
31,/home/anurag/opensource/TEST/dulwich-1.1.0/.git/objects/0f/dd52995b5301ea8e0...,4,0,0,0,0,0,0,4,,,2607
11,/home/anurag/opensource/TEST/dulwich-1.1.0/.git/config,0,0,0,0,3,0,0,3,,,1208
54,/home/anurag/opensource/TEST/dulwich-1.1.0/.git/objects/2e,0,0,3,0,0,0,0,3,EEXIST,File exists,2600
171,/home/anurag/opensource/TEST/dulwich-1.1.0/.git/objects/b6/5cbc7d0ddd08151bf...,0,0,0,0,3,0,0,3,,,1244
